# BWM
---
_**B**est **W**orst **M**ethod_

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

## 1. Introduction
---
The **BWM** (_Best Worst Method_) derives **criteria weights** from two sets of pairwise judgments: the **best criterion compared with the others**, and the **others compared with the worst criterion**.

This notebook implements the **linear BWM** proposed by [Rezaei (2016)](https://doi.org/10.1016/j.omega.2015.12.001). It minimizes discrepancies between weighted comparisons. The original [BWM (2015)](https://doi.org/10.1016/j.omega.2014.11.009) minimizes discrepancies between **weight ratios**, using a different optimization model.

BWM determines weights. To demonstrate their use, we then apply a **weighted sum model** to alternatives with explicitly defined performance scales.

### 1.1. Applications
---
The _BWM method_ can help determine criteria weights in decisions such as:

- Supplier selection
- Environmental investment planning
- Transport service evaluation
- Project prioritization

Consider a school district comparing **4 meal suppliers**. All suppliers already meet the required food safety and nutritional standards. The remaining decision considers **cost per meal** (BRL), **food safety audit score** (0–100), **on-time deliveries** (%), and **reusable packaging** (% of meals).

The following data, supplier names, and preference judgments are **illustrative**, representing a plausible procurement decision rather than an actual tender. **Cost** is minimized; the remaining criteria are maximized.

In [2]:
df = pd.DataFrame({
    "cost": [8.8, 8.0, 9.2, 7.5],
    "safety": [98, 91, 95, 84],
    "punctuality": [98, 97, 99, 92],
    "packaging": [70, 30, 95, 20]
})

df.index = [
    "River Cooperative",
    "Central Foods",
    "Green Kitchen",
    "Budget Meals"
]

df.style.background_gradient()

,cost,safety,punctuality,packaging
River Cooperative,8.800000,98,98,70
Central Foods,8.000000,91,97,30
Green Kitchen,9.200000,95,99,95
Budget Meals,7.500000,84,92,20


## 2. Method steps
---

### 2.1. Criteria, objectives, and best/worst criteria
---
Define the criteria and their performance scales before eliciting weights. Here, the district uses **linear value scales** over the agreed anchors below. Each comparison concerns improving one criterion from its **least preferred anchor** to its **most preferred anchor**, with the others unchanged.

Identify the **best** criterion $B$ (most important) and **worst** criterion $W$ (least important). These refer to criteria, not to supplier performances. The district chooses **safety** as best and **packaging** as worst.

In [3]:
CRITERIA = df.columns

OBJECTIVES = pd.Series(["min", "max", "max", "max"], index=CRITERIA)

LOWER = pd.Series([7, 80, 90, 0], index=CRITERIA)
UPPER = pd.Series([10, 100, 100, 100], index=CRITERIA)

BEST = "safety"
WORST = "packaging"

pd.DataFrame({
    "objective": OBJECTIVES,
    "lower anchor": LOWER,
    "upper anchor": UPPER
})

,objective,lower anchor,upper anchor
cost,min,7,10
safety,max,80,100
punctuality,max,90,100
packaging,max,0,100


### 2.2. Best-to-Others comparisons
---
Compare the best criterion with each criterion using the **1–9 scale**: **1** means equal importance, **3** moderate, **5** strong, **7** very strong, and **9** extreme importance; **2, 4, 6, 8** are intermediate values.

The **Best-to-Others vector** is:

$$ \large
A_B = (a_{B1}, \ldots, a_{Bn}), \qquad a_{BB}=1
$$

Here, safety is judged **3 times as important as cost**, **2 times as important as punctuality**, and **8 times as important as packaging**.

In [4]:
A_B = pd.Series([3, 1, 2, 8], index=CRITERIA, dtype=float)

A_B.to_frame("Best-to-Others")

,Best-to-Others
cost,3.0
safety,1.0
punctuality,2.0
packaging,8.0


### 2.3. Others-to-Worst comparisons
---
Compare each criterion with the worst criterion using the same scale:

$$ \large
A_W = (a_{1W}, \ldots, a_{nW})^T, \qquad a_{WW}=1
$$

The **best-to-worst** judgment appears in both vectors and must agree: $a_{BW}=8$. The two self-comparisons are fixed at 1. Therefore, only **$2n-3$ distinct judgments** are elicited: **5** for these **4 criteria**.

In [5]:
A_W = pd.Series([3, 8, 4, 1], index=CRITERIA, dtype=float)

if A_B[WORST] != A_W[BEST]:
    raise ValueError("The two best-to-worst judgments must agree.")

pd.DataFrame({"Best-to-Others": A_B, "Others-to-Worst": A_W})

,Best-to-Others,Others-to-Worst
cost,3.0,3.0
safety,1.0,8.0
punctuality,2.0,4.0
packaging,8.0,1.0


### 2.4. Linear optimization model
---
Find nonnegative weights that sum to one, minimizing the **largest absolute comparison discrepancy** $\xi_L$:

$$ \large
\begin{aligned}
\min \quad & \xi_L \\
\text{subject to} \quad
& |w_B-a_{Bj}w_j| \leq \xi_L, &&j=1,\ldots,n, \\
& |w_j-a_{jW}w_W| \leq \xi_L, &&j=1,\ldots,n, \\
& \sum_{j=1}^n w_j=1, \qquad w_j \geq 0, \qquad \xi_L \geq 0.
\end{aligned}
$$

Each absolute-value constraint becomes **two linear inequalities**. For example, $|w_B-a_{Bj}w_j|\leq\xi_L$ becomes:

$$ \large
w_B-a_{Bj}w_j-\xi_L \leq 0, \qquad
-w_B+a_{Bj}w_j-\xi_L \leq 0
$$

Use the variable order **$(w_1,\ldots,w_n,\xi_L)$**. The following table shows all inequality coefficients; each row has a right-hand side of **0**. Self-comparisons and the repeated best-to-worst comparison give redundant but harmless constraints.

The objective minimizes **xi_L**; the equality constraint makes the weights sum to **1**. Use SciPy's **HiGHS** solver.

In [6]:
n = len(CRITERIA)
b = CRITERIA.get_loc(BEST)
w = CRITERIA.get_loc(WORST)

A_ub = []

for j, criterion in enumerate(CRITERIA):
    # Best-to-Others: w_B - a_Bj*w_j
    row = np.zeros(n + 1)
    row[b] += 1
    row[j] -= A_B[criterion]

    for sign in [1, -1]:
        constraint = sign*row
        constraint[-1] = -1
        A_ub.append(constraint)

    # Others-to-Worst: w_j - a_jW*w_W
    row = np.zeros(n + 1)
    row[j] += 1
    row[w] -= A_W[criterion]

    for sign in [1, -1]:
        constraint = sign*row
        constraint[-1] = -1
        A_ub.append(constraint)

A_ub = np.array(A_ub)
b_ub = np.zeros(len(A_ub))

pd.DataFrame(
    A_ub,
    columns=[f'w_{criterion}' for criterion in CRITERIA] + ["xi_L"]
)

,w_cost,w_safety,w_punctuality,w_packaging,xi_L
0,-3.0,1.0,0.0,0.0,-1.0
1,3.0,-1.0,-0.0,-0.0,-1.0
2,1.0,0.0,0.0,-3.0,-1.0
3,-1.0,-0.0,-0.0,3.0,-1.0
4,0.0,0.0,0.0,0.0,-1.0
5,-0.0,-0.0,-0.0,-0.0,-1.0
6,0.0,1.0,0.0,-8.0,-1.0
7,-0.0,-1.0,-0.0,8.0,-1.0
8,0.0,1.0,-2.0,0.0,-1.0
9,-0.0,-1.0,2.0,-0.0,-1.0


In [7]:
c = np.zeros(n + 1)
c[-1] = 1

A_eq = np.ones((1, n + 1))
A_eq[0, -1] = 0

result = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=[1],
    bounds=[(0, None)]*(n + 1),
    method="highs"
)

if not result.success:
    raise RuntimeError(result.message)

pd.Series({
    "success": result.success,
    "xi_L": result.fun,
    "sum of weights": result.x[:-1].sum()
})

success               True
xi_L              0.012346
sum of weights         1.0
dtype: object

### 2.5. Criteria weights and consistency
---
Read the optimized **criteria weights** and **xi_L** from the solution. The weights already sum to one; no additional normalization is needed.

In [8]:
WEIGHTS = pd.Series(result.x[:-1], index=CRITERIA)
xi_L = float(result.x[-1])

WEIGHTS.to_frame("weight").style.background_gradient()

,weight
cost,0.172840
safety,0.506173
punctuality,0.259259
packaging,0.061728


Fully consistent judgments satisfy **$a_{Bj} \cdot a_{jW}=a_{BW}$** for every criterion. The fitted weights then reproduce both vectors exactly, with $\xi_L^*=0$.

For the linear model, $\xi_L^*$ measures the smallest achievable **maximum error in the weighted comparisons**. It is **not a percentage error in the weight ratios**. As clarified in the [author's tutorial](https://bestworstmethod.com/wp-content/uploads/2020/01/Best-Worst-Method-BWM-2019.pdf), do not divide it by the consistency-index table for the original nonlinear BWM.

Display both the input consistency check and the absolute residuals:

In [9]:
df_consistency = pd.DataFrame({
    "a_Bj * a_jW": A_B*A_W,
    "a_BW": A_B[WORST],
    "xi_L": xi_L,
    "BO residual": abs(WEIGHTS[BEST] - A_B*WEIGHTS),
    "OW residual": abs(WEIGHTS - A_W*WEIGHTS[WORST])
})

max_residual = df_consistency[["BO residual", "OW residual"]].to_numpy().max()

if max_residual > xi_L + 1e-9:
    raise ValueError("The fitted weights violate a comparison constraint.")

df_consistency

,a_Bj * a_jW,a_BW,xi_L,BO residual,OW residual
cost,9.0,8.0,0.012346,0.012346,0.012346
safety,8.0,8.0,0.012346,0.000000,0.012346
punctuality,8.0,8.0,0.012346,0.012346,0.012346
packaging,8.0,8.0,0.012346,0.012346,0.000000


The **cost** judgments are not fully consistent: $3\times3=9$, whereas the direct best-to-worst judgment is **8**. Accordingly, the optimized residual is positive: **$\xi_L^*\approx0.012346$**. The model finds a compromise, with approximately **50.62%** of the weight assigned to safety, **25.93%** to punctuality, **17.28%** to cost, and **6.17%** to packaging.

No automatic acceptance threshold is assumed. The table identifies the judgments to review; the following application uses them unchanged.

### 2.6. Application
---
Apply the BWM weights in a **weighted sum model**, using the agreed lower and upper anchors $l_j$ and $h_j$:

$$ \large
u_{ij}=\begin{cases}
\dfrac{x_{ij}-l_j}{h_j-l_j}, & \text{if criterion }j\text{ is maximized}, \\
\dfrac{h_j-x_{ij}}{h_j-l_j}, & \text{if criterion }j\text{ is minimized}.
\end{cases}
$$

These linear value functions assign **0** to the least preferred anchor and **1** to the most preferred anchor. All example values are within the ranges. This aggregation assumes that the criterion values can be combined additively, with compensation between criteria.

In [10]:
if not ((df >= LOWER) & (df <= UPPER)).all().all():
    raise ValueError("A performance lies outside the agreed evaluation range.")

df_a = (df - LOWER)/(UPPER - LOWER)

for criterion, objective in OBJECTIVES.items():
    if objective == "min":
        df_a[criterion] = 1 - df_a[criterion]

df_a.style.background_gradient(vmin=0, vmax=1)

,cost,safety,punctuality,packaging
River Cooperative,0.400000,0.900000,0.800000,0.700000
Central Foods,0.666667,0.550000,0.700000,0.300000
Green Kitchen,0.266667,0.750000,0.900000,0.950000
Budget Meals,0.833333,0.200000,0.200000,0.200000


Multiply each value by its criterion weight. Each row's contributions sum to the overall score:

$$ \large
S_i=\sum_{j=1}^n w_j u_{ij}
$$

In [11]:
df_contributions = df_a*WEIGHTS

df_contributions["score"] = df_contributions.sum(axis=1)

df_contributions

,cost,safety,punctuality,packaging,score
River Cooperative,0.069136,0.455556,0.207407,0.043210,0.775309
Central Foods,0.115226,0.278395,0.181481,0.018519,0.593621
Green Kitchen,0.046091,0.379630,0.233333,0.058642,0.717695
Budget Meals,0.144033,0.101235,0.051852,0.012346,0.309465


### 2.7. Score and Ranking
---
Rank the suppliers by **decreasing score**. Equal scores would share a rank; there are no ties in this example.

In [12]:
df_score = df.copy()

df_score["score"] = df_contributions["score"]
df_score["ranking"] = (
    df_score["score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

(
    df_score.sort_values("ranking")
    .style.background_gradient(subset=list(CRITERIA))
)

,cost,safety,punctuality,packaging,score,ranking
River Cooperative,8.800000,98,98,70,0.775309,1
Green Kitchen,9.200000,95,99,95,0.717695,2
Central Foods,8.000000,91,97,30,0.593621,3
Budget Meals,7.500000,84,92,20,0.309465,4


**River Cooperative**'s stronger safety audit score offsets Green Kitchen's better punctuality and packaging, with a lower meal cost also helping River.

Budget Meals offers the lowest price, but cost has less weight than safety and punctuality. This demonstrates how the elicited priorities affect the recommendation.

## 3. Examples
---

### 3.1. Water conservation investment
---
A factory compares **3 water conservation projects**. The criteria are **annual water savings** (m³/year, maximize), **simple payback period** (years, minimize), and **installation downtime** (days, minimize).

The data and judgments are **illustrative**. Use linear value scales over **0–12,000 m³/year**, **0–10 years**, and **0–20 days**. The manager compares improvements across these ranges, choosing water savings as **best** and downtime as **worst**.

Here, the Best-to-Others vector **(1, 2, 4)** and Others-to-Worst vector **(4, 2, 1)** are **fully consistent**. This provides a second application and an exact-fit example of BWM.

In [13]:
df_ex1 = pd.DataFrame({
    "water savings": [11000, 7000, 4500],
    "payback": [3.5, 2.5, 4.0],
    "downtime": [8, 4, 2]
})

df_ex1.index = [
    "Cooling loop upgrade",
    "Rinse-water recovery",
    "Rainwater storage"
]

df_ex1.style.background_gradient()

,water savings,payback,downtime
Cooling loop upgrade,11000,3.500000,8
Rinse-water recovery,7000,2.500000,4
Rainwater storage,4500,4.000000,2


In [14]:
CRITERIA_ex1 = df_ex1.columns

BEST_ex1 = "water savings"
WORST_ex1 = "downtime"

A_B_ex1 = pd.Series([1, 2, 4], index=CRITERIA_ex1, dtype=float)
A_W_ex1 = pd.Series([4, 2, 1], index=CRITERIA_ex1, dtype=float)

pd.DataFrame({
    "Best-to-Others": A_B_ex1,
    "Others-to-Worst": A_W_ex1,
    "a_Bj * a_jW": A_B_ex1*A_W_ex1
})

,Best-to-Others,Others-to-Worst,a_Bj * a_jW
water savings,1.0,4.0,4.0
payback,2.0,2.0,4.0
downtime,4.0,1.0,4.0


In [15]:
n_ex1 = len(CRITERIA_ex1)
b_ex1 = CRITERIA_ex1.get_loc(BEST_ex1)
w_ex1 = CRITERIA_ex1.get_loc(WORST_ex1)

A_ub_ex1 = []

for j, criterion in enumerate(CRITERIA_ex1):
    # Best-to-Others constraints
    row = np.zeros(n_ex1 + 1)
    row[b_ex1] += 1
    row[j] -= A_B_ex1[criterion]

    for sign in [1, -1]:
        constraint = sign*row
        constraint[-1] = -1
        A_ub_ex1.append(constraint)

    # Others-to-Worst constraints
    row = np.zeros(n_ex1 + 1)
    row[j] += 1
    row[w_ex1] -= A_W_ex1[criterion]

    for sign in [1, -1]:
        constraint = sign*row
        constraint[-1] = -1
        A_ub_ex1.append(constraint)

c_ex1 = np.zeros(n_ex1 + 1)
c_ex1[-1] = 1

A_eq_ex1 = np.ones((1, n_ex1 + 1))
A_eq_ex1[0, -1] = 0

result_ex1 = linprog(
    c_ex1,
    A_ub=np.array(A_ub_ex1),
    b_ub=np.zeros(len(A_ub_ex1)),
    A_eq=A_eq_ex1,
    b_eq=[1],
    bounds=[(0, None)]*(n_ex1 + 1),
    method="highs"
)

if not result_ex1.success:
    raise RuntimeError(result_ex1.message)

WEIGHTS_ex1 = pd.Series(result_ex1.x[:-1], index=CRITERIA_ex1)
xi_L_ex1 = float(result_ex1.x[-1])

pd.DataFrame({
    "weight": WEIGHTS_ex1,
    "xi_L": xi_L_ex1,
    "BO residual": abs(WEIGHTS_ex1[BEST_ex1] - A_B_ex1*WEIGHTS_ex1),
    "OW residual": abs(WEIGHTS_ex1 - A_W_ex1*WEIGHTS_ex1[WORST_ex1])
})

,weight,xi_L,BO residual,OW residual
water savings,0.571429,0.0,0.000000e+00,0.000000e+00
payback,0.285714,0.0,1.110223e-16,5.551115e-17
downtime,0.142857,0.0,0.000000e+00,0.000000e+00


The model recovers **4/7**, **2/7**, and **1/7** for water savings, payback, and downtime, with **xi_L = 0** up to numerical precision. The weight ratios reproduce every judgment exactly.

Normalize the project performances using the stated anchors and display their **weighted contributions**.

In [16]:
OBJECTIVES_ex1 = pd.Series(
    ["max", "min", "min"], index=CRITERIA_ex1
)

LOWER_ex1 = pd.Series([0, 0, 0], index=CRITERIA_ex1)
UPPER_ex1 = pd.Series([12000, 10, 20], index=CRITERIA_ex1)

df_ex1_a = (df_ex1 - LOWER_ex1)/(UPPER_ex1 - LOWER_ex1)

for criterion, objective in OBJECTIVES_ex1.items():
    if objective == "min":
        df_ex1_a[criterion] = 1 - df_ex1_a[criterion]

df_ex1_a.style.background_gradient(vmin=0, vmax=1)

,water savings,payback,downtime
Cooling loop upgrade,0.916667,0.650000,0.600000
Rinse-water recovery,0.583333,0.750000,0.800000
Rainwater storage,0.375000,0.600000,0.900000


In [17]:
df_ex1_contributions = df_ex1_a*WEIGHTS_ex1

df_ex1_contributions["score"] = df_ex1_contributions.sum(axis=1)

df_ex1_contributions

,water savings,payback,downtime,score
Cooling loop upgrade,0.523810,0.185714,0.085714,0.795238
Rinse-water recovery,0.333333,0.214286,0.114286,0.661905
Rainwater storage,0.214286,0.171429,0.128571,0.514286


In [18]:
df_ex1_score = df_ex1.copy()

df_ex1_score["score"] = df_ex1_contributions["score"]
df_ex1_score["ranking"] = (
    df_ex1_score["score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

(
    df_ex1_score.sort_values("ranking")
    .style.background_gradient(subset=list(CRITERIA_ex1))
)

,water savings,payback,downtime,score,ranking
Cooling loop upgrade,11000,3.500000,8,0.795238,1
Rinse-water recovery,7000,2.500000,4,0.661905,2
Rainwater storage,4500,4.000000,2,0.514286,3


The **cooling loop upgrade**'s larger water savings outweigh the other projects' shorter payback or installation time under the manager's priorities.

## 4. References
---

- Rezaei, J. (2015). [*Best-worst multi-criteria decision-making method*](https://doi.org/10.1016/j.omega.2014.11.009).
- Rezaei, J. (2016). [*Best-worst multi-criteria decision-making method: Some properties and a linear model*](https://doi.org/10.1016/j.omega.2015.12.001).
- Rezaei, J. (2019). [*BWM: Best Worst Method*](https://bestworstmethod.com/wp-content/uploads/2020/01/Best-Worst-Method-BWM-2019.pdf). Author's tutorial; slides 20–21 explain the linear formulation and its consistency indicator.